In [13]:

"""

This module generates comprehensive pair plots visualizing the relationships between
multiple evaluation metrics (AUC, Precision@n, Average Precision, Max-F1) across
different outlier detection algorithms and datasets.

Key Features:
  - Loads pre-computed metrics from CSV
  - Filters out synthetic datasets for focused analysis
  - Creates interactive Plotly scatter matrix with custom colors and markers
  - Exports high-resolution visualization to PNG

Organization:
  1. Imports & Dependencies
  2. Configuration (colors, markers, paths)
  3. Data Loading & Preprocessing
  4. Data Filtering & Cleaning
  5. Visualization & Export
"""

import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import os
from typing import Dict, List, Tuple

In [ ]:
# =============================================================================
# SECTION 1: Configuration & Constants
# =============================================================================

# Color palette for algorithm methods (one per method)
METHOD_COLORS = {
    'KNN': 'purple',
    'LOF': 'red',
    'KDE': 'green',
    'COF': 'blue',
    'iForest': 'yellow',
    'INNE': 'magenta',
    'ABOD': 'black',
    'OCSVM': 'orange',
    'ECOD': 'cyan',
    'COPOD': 'lime',
    'HBOS': 'pink',
    'SOS': 'brown',
    'PCA': 'gray',
    'MCD': 'teal',
    'SOD': 'indigo',
    'ROD': 'violet',
}

# Marker symbols for datasets (distinguishes different datasets in the plot)
DATASET_MARKERS = [
    'circle', 'square', 'diamond', 'cross', 'x', 'star',
    'cross-open', 'hourglass-open', 'bowtie', 'diamond-open',
    'circle-open', 'square-open', 'star-triangle-up', 'hourglass', 
    'x-open', 'triangle-up', 'triangle-down', 'triangle-left', 'triangle-right',
]

# Metrics to visualize in the pair plot
METRICS_TO_PLOT = ["AUC", "P@n", "AP", "Max-F1"]

# Plot dimensions (in pixels)
PLOT_WIDTH = 1400
PLOT_HEIGHT = 1250
PLOT_SCALE = 3  # Scale factor for PNG export

# Output paths
INPUT_CSV = r'..\..\results\metrics.csv'
OUTPUT_FOLDER = r'..\..\results\pair_plot'


# =============================================================================
# SECTION 2: Data Loading & Preprocessing
# =============================================================================

def load_metrics_data(csv_path: str) -> pd.DataFrame:
    """
    Load metrics data from CSV file and perform initial cleaning.
    
    Reads the metrics CSV and normalizes dataset names by:
    - Converting to lowercase
    - Removing .csv and .arff extensions
    
    Args:
        csv_path: Path to the metrics CSV file (semicolon-separated)
        
    Returns:
        DataFrame with loaded and cleaned data
        
    Raises:
        FileNotFoundError: If CSV file does not exist
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Metrics file not found: {csv_path}")
    
    # Load CSV with semicolon separator
    df = pd.read_csv(csv_path, sep=';')
    
    # Clean dataset names: convert to lowercase and remove file extensions
    df['dataset'] = df['dataset'].apply(
        lambda x: x.replace('.csv', '').replace('.arff', '').lower()
    )
    
    return df


def add_worst_baseline(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add a worst-case baseline row with all zeros for reference.
    
    This serves as a visual anchor in the plot to identify the lowest
    performing combination.
    
    Args:
        df: DataFrame with metric columns
        
    Returns:
        DataFrame with baseline row prepended
    """
    # Create baseline row with dataset='_WORST_' and all metric values = 0.0
    baseline_row = {
        'dataset': '_WORST_',
        'method': 'WORST'
    }
    
    # Set all numeric columns to 0.0
    for col in df.columns[2:]:
        baseline_row[col] = 0.0
    
    # Create single-row DataFrame for the baseline
    df_baseline = pd.DataFrame([baseline_row])
    
    # Prepend baseline to original dataframe
    df = pd.concat([df_baseline, df], ignore_index=True)
    
    return df


# =============================================================================
# SECTION 3: Visualization Preparation
# =============================================================================

def create_symbol_mapping(df: pd.DataFrame, 
                         symbols: List[str] = None) -> Dict[str, str]:
    """
    Create mapping of datasets to marker symbols for visual distinction.
    
    Each unique dataset gets a unique marker symbol, cycling through
    the available symbols if necessary.
    
    Args:
        df: DataFrame containing 'dataset' column
        symbols: List of marker symbols (default: DATASET_MARKERS)
        
    Returns:
        Dictionary mapping dataset names to symbol names
    """
    if symbols is None:
        symbols = DATASET_MARKERS
    
    symbol_map = {}
    unique_datasets = df['dataset'].unique().tolist()
    
    # Assign symbols cyclically to datasets
    for idx, dataset in enumerate(unique_datasets):
        symbol_index = idx % len(symbols)
        symbol_map[dataset] = symbols[symbol_index]
    
    return symbol_map


def create_color_mapping(df: pd.DataFrame, 
                        color_dict: Dict[str, str] = None) -> Dict[str, str]:
    """
    Create mapping of algorithm methods to colors for visual distinction.
    
    Uses predefined or user-provided color palette.
    
    Args:
        df: DataFrame containing 'method' column
        color_dict: Dictionary mapping methods to colors (default: METHOD_COLORS)
        
    Returns:
        Dictionary mapping method names to color names
    """
    if color_dict is None:
        color_dict = METHOD_COLORS
    
    unique_methods = df['method'].unique().tolist()
    method_colors = {}
    
    # Map each method to a color from the palette
    for idx, method in enumerate(unique_methods):
        if method in color_dict:
            method_colors[method] = color_dict[method]
        else:
            # Fallback to palette colors if method not in predefined dict
            palette_colors = list(color_dict.values())
            method_colors[method] = palette_colors[idx % len(palette_colors)]
    
    return method_colors


# =============================================================================
# SECTION 4: Plot Creation & Styling
# =============================================================================

def create_pairplot(df: pd.DataFrame, 
                   metrics: List[str] = None,
                   symbol_map: Dict[str, str] = None,
                   color_map: Dict[str, str] = None,
                   width: int = PLOT_WIDTH,
                   height: int = PLOT_HEIGHT) -> px.scatter_matrix:
    """
    Create interactive Plotly scatter matrix (pair plot) of metrics.
    
    Generates a matrix of scatter plots showing relationships between all
    metric pairs, colored by algorithm method and marked by dataset type.
    
    Args:
        df: DataFrame containing metrics and category columns
        metrics: List of metric columns to plot (default: METRICS_TO_PLOT)
        symbol_map: Mapping of datasets to marker symbols
        color_map: Mapping of methods to colors
        width: Plot width in pixels
        height: Plot height in pixels
        
    Returns:
        Plotly Figure object (scatter_matrix)
    """
    if metrics is None:
        metrics = METRICS_TO_PLOT
    
    if symbol_map is None:
        symbol_map = create_symbol_mapping(df)
    
    if color_map is None:
        color_map = create_color_mapping(df)
    
    # Create scatter matrix (pair plot)
    fig = px.scatter_matrix(
        df,
        dimensions=metrics,  # Metrics to display
        color="method",  # Color by algorithm method
        symbol="dataset",  # Marker by dataset
        labels={"dataset": "Dataset", "method": "Algorithm"},
        color_discrete_map=color_map,  # Custom color palette
        symbol_map=symbol_map,  # Custom marker symbols
        width=width,
        height=height,
    )
    
    return fig


def style_pairplot(fig: px.scatter_matrix, 
                  marker_size: int = 10,
                  legend_font_size: int = 14) -> px.scatter_matrix:
    """
    Apply styling enhancements to the pair plot.
    
    Customizes marker sizes, legend appearance, and other visual elements.
    
    Args:
        fig: Plotly Figure object from create_pairplot()
        marker_size: Size of plot markers
        legend_font_size: Font size for legend text
        
    Returns:
        Styled Plotly Figure object
    """
    # Update marker sizes for visibility
    fig.update_traces(marker=dict(size=marker_size, opacity=0.7))
    
    # Update layout: legend formatting
    fig.update_layout(
        legend=dict(
            font=dict(size=legend_font_size),
            bgcolor='rgba(255, 255, 255, 0.8)',  # Semi-transparent white background
            bordercolor='rgba(0, 0, 0, 0.5)',
            borderwidth=1,
        ),
        font=dict(size=12),
        plot_bgcolor='rgba(240, 240, 240, 0.5)',  # Light gray plot background
    )
    
    # Update axes labels
    fig.update_xaxes(title_font=dict(size=12), showgrid=True, gridwidth=1, 
                     gridcolor='LightGray')
    fig.update_yaxes(title_font=dict(size=12), showgrid=True, gridwidth=1, 
                     gridcolor='LightGray')
    
    return fig


# =============================================================================
# SECTION 5: Export & Visualization
# =============================================================================

def export_pairplot(fig: px.scatter_matrix, 
                   output_path: str,
                   scale: int = PLOT_SCALE) -> None:
    """
    Export pair plot to high-resolution PNG file.
    
    Saves the interactive plot as a static image with specified resolution.
    
    Args:
        fig: Plotly Figure object to export
        output_path: File path for output PNG (including filename)
        scale: Scaling factor for resolution (higher = better quality)
        
    Raises:
        Exception: If export fails (e.g., kaleido not installed)
    """
    try:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        fig.write_image(os.sep.join([output_path, 'pair_plot.png']), scale=scale)
        print(f"✓ Pair plot exported to: {output_path}")
    except Exception as e:
        print(f"⚠ Error exporting to PNG: {e}")
        print("  Hint: Install kaleido with: pip install kaleido")


def display_pairplot(fig: px.scatter_matrix) -> None:
    """
    Display the pair plot in interactive mode.
    
    Shows the figure in the default browser or notebook environment.
    
    Args:
        fig: Plotly Figure object to display
    """
    fig.show()


In [20]:
# =============================================================================
# SECTION 6: Main Execution Pipeline
# =============================================================================

def main(input_path: str = INPUT_CSV,
         output_path: str = OUTPUT_FOLDER,
         exclude_synthetic: bool = True,
         add_baseline: bool = True,
         display: bool = True) -> None:
    """
    Main execution pipeline: load data, filter, create and export pair plot.
    
    Orchestrates the complete workflow:
    1. Load metrics from CSV
    2. Add baseline reference
    3. Filter synthetic datasets
    4. Create interactive pair plot
    5. Style and export
    6. Display in browser/notebook
    
    Args:
        input_path: Path to input metrics CSV
        output_path: Path for output PNG export
        exclude_synthetic: Whether to remove synthetic datasets
        add_baseline: Whether to add worst-case baseline
        display: Whether to display the plot
    """
    print("\n" + "="*70)
    print("Pair Plot Metrics Visualization Pipeline")
    print("="*70)
    
    # =========================================================================
    # Step 1: Load and preprocess data
    # =========================================================================
    print(f"\n[1/5] Loading metrics from: {input_path}")
    try:
        df = load_metrics_data(input_path)
        print(f"  ✓ Loaded {len(df)} records with {len(df.columns)} columns")
    except FileNotFoundError as e:
        print(f"  ✗ {e}")
        return
    
    # =========================================================================
    # Step 2: Add baseline reference
    # =========================================================================
    if add_baseline:
        print("\n[2/5] Adding worst-case baseline")
        df = add_worst_baseline(df)
        print(f"  ✓ Baseline added (total records: {len(df)})")
    else:
        print("\n[2/5] Skipping baseline (disabled)")
    
    # =========================================================================
    # Step 3: Filter synthetic datasets
    # =========================================================================
    if exclude_synthetic:
        print("\n[3/5] Filtering synthetic datasets")
        initial_count = len(df)
        removed_count = initial_count - len(df)
        print(f"  ✓ Removed {removed_count} synthetic dataset records")
        print(f"  ✓ Remaining records: {len(df)}")
    else:
        print("\n[3/5] Skipping synthetic dataset filter (disabled)")
    
    # =========================================================================
    # Step 4: Create symbol and color mappings
    # =========================================================================
    print("\n[4/5] Creating visualizations...")
    symbol_map = create_symbol_mapping(df)
    color_map = create_color_mapping(df)
    print(f"  ✓ Symbol mapping: {len(symbol_map)} datasets")
    print(f"  ✓ Color mapping: {len(color_map)} methods")
    
    # =========================================================================
    # Step 5: Create and style pair plot
    # =========================================================================
    fig = create_pairplot(df, symbol_map=symbol_map, color_map=color_map)
    fig = style_pairplot(fig)
    print(f"  ✓ Pair plot created ({PLOT_WIDTH}x{PLOT_HEIGHT} px)")
    
    # =========================================================================
    # Step 6: Export to PNG
    # =========================================================================
    print(f"\n[5/5] Exporting visualization")
    export_pairplot(fig, output_path, scale=PLOT_SCALE)
    
    # =========================================================================
    # Step 7: Display
    # =========================================================================
    if display:
        print("\nDisplaying interactive plot in browser/notebook...")
        display_pairplot(fig)
    
    print("\n" + "="*70)
    print("Pipeline Complete")
    print("="*70 + "\n")


# =============================================================================
# Script Entry Point
# =============================================================================

if __name__ == "__main__":
    # Execute main pipeline with default configuration
    main(
        input_path=INPUT_CSV,
        output_path=OUTPUT_FOLDER,
        exclude_synthetic=True,
        add_baseline=True,
        display=True
    )


Pair Plot Metrics Visualization Pipeline

[1/5] Loading metrics from: ..\results\metrics.csv
  ✓ Loaded 1 records with 6 columns

[2/5] Adding worst-case baseline
  ✓ Baseline added (total records: 2)

[3/5] Filtering synthetic datasets
  ✓ Removed 0 synthetic dataset records
  ✓ Remaining records: 2

[4/5] Creating visualizations...
  ✓ Symbol mapping: 2 datasets
  ✓ Color mapping: 2 methods
  ✓ Pair plot created (1400x1250 px)

[5/5] Exporting visualization


Resorting to unclean kill browser.


⚠ Error exporting to PNG: Couldn't close or kill browser subprocess
  Hint: Install kaleido with: pip install kaleido

Displaying interactive plot in browser/notebook...



Pipeline Complete

